<a href="https://colab.research.google.com/github/julia-zhmurko/JuliaZhmurko-ML/blob/main/%D0%9B%D0%B0%D0%B1%D0%BE%D1%80%D0%B0%D1%82%D0%BE%D1%80%D0%BD%D1%96_%D1%80%D0%BE%D0%B1%D0%BE%D1%82%D0%B8/%D0%96%D0%BC%D1%83%D1%80%D0%BA%D0%BE_%D0%AE%D0%BB%D1%96%D1%8F_%D0%A4%D0%86%D0%A2_4_10_%D0%9B%D0%A0_9_%D0%A0%D0%B5%D0%BA%D0%BE%D0%BC%D0%B5%D0%BD%D0%B4%D0%B0%D1%86%D1%96%D0%B9%D0%BD%D1%96_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Жмурко Юлія Андріївна

**Група:** ФІТ 4-10

**Дата:** 18.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

**Виконала:** студентка групи ФІТ 4-10 Жмурко Юлія Андріївна

**Була присутня на парі (18.11.2026)**

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [18]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

###**Пояснення:**

Бібліотека `NumPy` використовується для роботи з числовими масивами, `SciPy` – для наукових обчислень, `scikit-learn` – для машинного навчання та розбиття даних, `pandas` – для обробки й аналізу табличних даних. Бібліотека `scikit-surprise` призначена для побудови та оцінювання рекомендаційних моделей, зокрема алгоритмів `SVD` і `KNNBasic`. Для забезпечення сумісності бібліотек встановлюються конкретні версії пакетів.

In [19]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.15.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


###**Пояснення:**

Із бібліотеки `surprise` імпортуються засоби завантаження даних (`Dataset`, `Reader`), алгоритми побудови рекомендаційних моделей (`SVD`, `KNNBasic`) та модуль `accuracy` для оцінювання точності прогнозування рейтингів. Також імпортуються `GridSearchCV` для підбору оптимальних гіперпараметрів моделей і `train_test_split` для розбиття даних на навчальну та тестову вибірки.

## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [20]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


###**Пояснення:**

Завантаження вбудованого набору даних `MovieLens 100K` із бібліотеки `Surprise` за допомогою методу `Dataset.load_builtin()`. Отримані дані перетворюються на таблицю `pandas.DataFrame`, яка містить чотири стовпці: `user_id` – ідентифікатор користувача, `item_id` – ідентифікатор фільму, `rating` – оцінка, виставлена користувачем, `timestamp` – час виставлення оцінки. За допомогою атрибута `shape` визначається розмір таблиці, а метод `head()` використовується для відображення перших п’яти записів. Набір даних містить 100 000 записів і 4 стовпці.



In [21]:
# TODO 2.1 — Дослідіть набір даних:
# - кількість унікальних користувачів;
# - кількість унікальних фільмів;
# - мінімальне, максимальне, середнє значення rating;
# - пропущені значення (за наявності).

# Ваш код:
print('Кількість унікальних користувачів:', ratings_df['user_id'].nunique())
print('Кількість унікальних фільмів:', ratings_df['item_id'].nunique())

print('Мінімальна оцінка:', ratings_df['rating'].min())
print('Максимальна оцінка:', ratings_df['rating'].max())
print('Середня оцінка:', round(ratings_df['rating'].mean(), 2))

print('\nКількість пропущених значень:')
display(ratings_df.isnull().sum())

Кількість унікальних користувачів: 943
Кількість унікальних фільмів: 1682
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.53

Кількість пропущених значень:


,0
user_id,0
item_id,0
rating,0
timestamp,0


###**Пояснення:**

Виконується первинний аналіз набору даних `MovieLens 100K` для визначення його основних характеристик. За допомогою методу `nunique()` обчислюється кількість унікальних користувачів і фільмів. Методи `min()`, `max()` та `mean()` використовуються для визначення мінімальної, максимальної та середньої оцінки відповідно. Середнє значення округлюється до двох знаків після коми за допомогою функції `round()`. Додатково методи `isnull().sum()` дозволяють перевірити наявність пропущених значень у кожному стовпці таблиці.

**Короткі спостереження щодо даних:**

_Запишіть 2–3 речення про кількість користувачів, фільмів та характер оцінок._

---
Набір даних `MovieLens 100K` містить 100 000 оцінок, виставлених 943 унікальними користувачами для 1682 фільмів. Оцінки перебувають у діапазоні від 1 до 5, а їхнє середнє значення становить 3,53, що свідчить про загалом позитивний характер оцінювання фільмів користувачами. Пропущених значень у наборі даних не виявлено, тому його можна використовувати для подальшого навчання рекомендаційних моделей.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [22]:
# TODO 3.1 — Створіть train_df та test_df із ratings_df.
# Використайте sklearn_train_test_split(..., test_size=0.2, random_state=42).

# Ваш код:
train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size = 0.2,
    random_state = 42
)

print('Розмір навчальної вибірки:', train_df.shape)
print('Розмір тестової вибірки:', test_df.shape)

Розмір навчальної вибірки: (80000, 4)
Розмір тестової вибірки: (20000, 4)


###**Пояснення:**

Виконується випадкове розбиття набору даних `MovieLens 100K` на навчальну та тестову вибірки за допомогою функції `sklearn_train_test_split()`. Параметр `test_size=0.2` визначає частку тестових даних у розмірі 20%, тоді як решта 80% використовується для навчання рекомендаційних моделей. Параметр `random_state=42` забезпечує відтворюваність результатів випадкового розбиття.

Навчальна вибірка `train_df` містить 80 000 записів і 4 стовпці, а тестова вибірка `test_df` – 20 000 записів і 4 стовпці.

In [23]:
# TODO 3.2 — Підготуйте об’єкти для Surprise:
# 1. reader = Reader(rating_scale=...)
#    Підказка: data.reader.rating_scale
# 2. train_data = Dataset.load_from_df(...)
#    Візьміть тільки user_id, item_id, rating із train_df.
# 3. testset = список кортежів (user_id, item_id, rating) із test_df.
# 4. Перевірте розміри train/test.

# Ваш код:
reader = Reader(rating_scale = data.reader.rating_scale)

train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader
)

trainset = train_data.build_full_trainset()

testset = list(
    test_df[['user_id', 'item_id', 'rating']].itertuples(
        index = False, name = None
    )
)

print('Кількість оцінок у навчальній вибірці:', trainset.n_ratings)
print('Кількість користувачів у навчальній вибірці:', trainset.n_users)
print('Кількість фільмів у навчальній вибірці:', trainset.n_items)
print('Кількість оцінок у тестовій вибірці:', len(testset))

Кількість оцінок у навчальній вибірці: 80000
Кількість користувачів у навчальній вибірці: 943
Кількість фільмів у навчальній вибірці: 1653
Кількість оцінок у тестовій вибірці: 20000


###**Пояснення:**

Виконується підготовка навчальної та тестової вибірок для подальшого використання в бібліотеці `Surprise`. За допомогою класу `Reader` задається шкала оцінювання фільмів від 1 до 5, яка відповідає початковому набору даних `MovieLens 100K`. Метод `Dataset.load_from_df()` перетворює навчальні дані зі стовпцями `user_id`, `item_id` та `rating` у формат бібліотеки `Surprise`, після чого метод `build_full_trainset()` формує навчальний набір `trainset`. Тестові дані за допомогою методу `itertuples()` перетворюються на список кортежів (`user_id`, `item_id`, `rating`), необхідний для подальшого оцінювання моделей.

Навчальний набір містить 80 000 оцінок, 943 унікальних користувачів та 1653 унікальних фільми. Тестовий набір містить 20 000 оцінок. Кількість фільмів у навчальній вибірці є меншою за загальну кількість фільмів у початковому наборі даних (1682), оскільки частина фільмів потрапила лише до тестової вибірки. Усі 943 користувачі представлені в навчальному наборі.

## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [24]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

print('Гіперпараметри моделі SVD:')
for param, values in param_grid_svd.items():
    print(f'{param}: {values}')

print('\nГіперпараметри моделі KNNBasic:')
for param, values in param_grid_knn.items():
    if param == 'sim_options':
        for option, values_option in values.items():
            print(f'{option}: {values_option}')
    else:
        print(f'{param}: {values}')

svd_combinations = 1
for values in param_grid_svd.values():
    svd_combinations *= len(values)

knn_combinations = 1
for param, values in param_grid_knn.items():
    if param == 'sim_options':
        for values_option in values.values():
            knn_combinations *= len(values_option)
    else:
        knn_combinations *= len(values)

print('\nКількість комбінацій для SVD:', svd_combinations)
print('Кількість комбінацій для KNNBasic:', knn_combinations)

Гіперпараметри моделі SVD:
n_factors: [50, 100]
lr_all: [0.005, 0.01]
reg_all: [0.02, 0.05]

Гіперпараметри моделі KNNBasic:
k: [20, 40]
min_k: [1, 5]
name: ['cosine', 'pearson']
user_based: [False, True]

Кількість комбінацій для SVD: 8
Кількість комбінацій для KNNBasic: 16


###**Пояснення:**

Виконується визначення початкових сіток гіперпараметрів для подальшого налаштування рекомендаційних моделей `SVD` та `KNNBasic`. Для моделі `SVD` задаються кількість прихованих факторів (`n_factors`), швидкість навчання (`lr_all`) та коефіцієнт регуляризації (`reg_all`), який використовується для зменшення ризику перенавчання. Для моделі `KNNBasic` визначаються максимальна кількість найближчих сусідів (`k`), мінімальна кількість сусідів (`min_k`), метрики схожості (`cosine`, `pearson`) та підхід до формування рекомендацій на основі схожості користувачів (`user_based=True`) або фільмів (`user_based=False`).

За допомогою циклів `for` та функції `print()` виводяться задані значення гіперпараметрів для кожної моделі. Додатково обчислюється кількість можливих комбінацій гіперпараметрів шляхом перемноження кількості значень кожного параметра.

Для моделі `SVD` передбачено 8 комбінацій гіперпараметрів, які охоплюють два значення кількості прихованих факторів (50, 100), два значення швидкості навчання (0,005; 0,010) та два значення регуляризації (0,02; 0,05). Для моделі `KNNBasic` передбачено 16 комбінацій гіперпараметрів, які включають різні значення кількості сусідів (20, 40), мінімальної кількості сусідів (1, 5), дві метрики схожості та два підходи до формування рекомендацій.

**Обґрунтування вибору параметрів:**

_Поясніть, які параметри впливають на якість моделей і чому обрали саме ці значення._

---

Для моделі `SVD` обрано кількість прихованих факторів `n_factors (50, 100)`, що визначає складність моделі та її здатність виявляти приховані вподобання користувачів. Параметр швидкості навчання `lr_all (0,005; 0,010)` впливає на швидкість оновлення параметрів моделі під час навчання, а коефіцієнт регуляризації `reg_all (0,02; 0,05)` допомагає зменшити ризик перенавчання. Обрані значення дозволяють порівняти моделі різної складності та дослідити вплив швидкості навчання й регуляризації на якість прогнозування.

Для моделі `KNNBasic` обрано максимальну кількість найближчих сусідів `k (20, 40)`, яка визначає кількість схожих користувачів або фільмів, що враховуються під час прогнозування. Параметр `min_k (1, 5)` задає мінімальну кількість сусідів, необхідну для обчислення прогнозу. Для оцінювання схожості обрано метрики `cosine` та `pearson`, які дозволяють порівняти різні способи визначення подібності між користувачами або фільмами. Параметр `user_based` дає змогу дослідити два підходи до формування рекомендацій: на основі схожості користувачів (`True`) та на основі схожості фільмів (`False`).

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [25]:
# TODO 5.1 — Підберіть гіперпараметри SVD.
# Підказка: GridSearchCV(SVD, param_grid_svd,
#                        measures=['rmse', 'mae'], cv=3, n_jobs=-1)
# Після створення викличте .fit(train_data).

# Ваш код:
grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures = ['rmse', 'mae'],
    cv = 3,
    n_jobs = -1
)

grid_search_svd.fit(train_data)

print('Пошук найкращих гіперпараметрів SVD завершено.')
print('Кількість перевірених комбінацій:', svd_combinations)
print('Кількість розбиттів перехресної перевірки:', 3)

Пошук найкращих гіперпараметрів SVD завершено.
Кількість перевірених комбінацій: 8
Кількість розбиттів перехресної перевірки: 3


###**Пояснення:**

Виконується автоматизований пошук найкращих гіперпараметрів рекомендаційної моделі `SVD` за допомогою `GridSearchCV` із бібліотеки `Surprise`. Для пошуку використовується раніше сформована сітка `param_grid_svd`, яка містить різні значення кількості прихованих факторів, швидкості навчання та коефіцієнта регуляризації. Якість прогнозування оцінюється за метриками `RMSE` та `MAE` із застосуванням трикратної перехресної перевірки (`cv=3`). Параметр `n_jobs=-1` дозволяє використовувати всі доступні процесорні ядра для паралельного виконання обчислень. Метод `fit(train_data)` запускає пошук оптимальних налаштувань виключно на навчальних даних.

Загалом перевірено 8 комбінацій гіперпараметрів, кожна з яких оцінювалася за допомогою трикратної перехресної перевірки.

In [26]:
# TODO 5.2 — Підберіть гіперпараметри KNNBasic.
# Підказка: використайте param_grid_knn та ті самі метрики/cv.

# Ваш код:
grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures = ['rmse', 'mae'],
    cv = 3,
    n_jobs = -1
)

grid_search_knn.fit(train_data)

print('Пошук найкращих гіперпараметрів KNNBasic завершено.')
print('Кількість перевірених комбінацій:', knn_combinations)
print('Кількість розбиттів перехресної перевірки:', 3)

Пошук найкращих гіперпараметрів KNNBasic завершено.
Кількість перевірених комбінацій: 16
Кількість розбиттів перехресної перевірки: 3


###**Пояснення:**

Виконується автоматизований пошук найкращих гіперпараметрів рекомендаційної моделі `KNNBasic` за допомогою `GridSearchCV`. Для пошуку використовується сітка `param_grid_knn`, яка містить різні значення максимальної та мінімальної кількості найближчих сусідів, метрики схожості (`cosine`, `pearson`) і підходи до формування рекомендацій на основі користувачів або фільмів. Для оцінювання якості прогнозування застосовуються метрики `RMSE` та `MAE` із трикратною перехресною перевіркою (`cv=3`). Параметр `n_jobs=-1` забезпечує паралельне виконання обчислень, а метод `fit(train_data)` здійснює пошук найкращих налаштувань лише на навчальній вибірці.

Загалом перевірено 16 комбінацій гіперпараметрів із використанням трикратної перехресної перевірки, що відповідає 48 навчанням та оцінюванням моделі.

In [27]:
# TODO 5.3 — Виведіть найкращий RMSE, MAE та параметри обох моделей.
# Підказки:
# grid_search_svd.best_score['rmse']
# grid_search_svd.best_params['rmse']
# grid_search_knn.best_score['rmse']

# Ваш код:
print('Найкращі результати моделі SVD:')
print('RMSE:', round(grid_search_svd.best_score['rmse'], 4))
print('MAE:', round(grid_search_svd.best_score['mae'], 4))
print('Найкращі параметри за RMSE:', grid_search_svd.best_params['rmse'])
print('Найкращі параметри за MAE:', grid_search_svd.best_params['mae'])

print('\nНайкращі результати моделі KNNBasic:')
print('RMSE:', round(grid_search_knn.best_score['rmse'], 4))
print('MAE:', round(grid_search_knn.best_score['mae'], 4))
print('Найкращі параметри за RMSE:', grid_search_knn.best_params['rmse'])
print('Найкращі параметри за MAE:', grid_search_knn.best_params['mae'])

Найкращі результати моделі SVD:
RMSE: 0.9442
MAE: 0.7444
Найкращі параметри за RMSE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}
Найкращі параметри за MAE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

Найкращі результати моделі KNNBasic:
RMSE: 1.0274
MAE: 0.8155
Найкращі параметри за RMSE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}
Найкращі параметри за MAE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


###**Пояснення:**

За допомогою атрибута `best_score` визначаються найменші середні значення метрик `RMSE` та `MAE`, отримані під час перехресної перевірки. Атрибут `best_params` використовується для відображення оптимальних комбінацій гіперпараметрів за кожною метрикою. Отримані значення округлюються до чотирьох знаків після коми за допомогою функції `round()`.

Для моделі `SVD` найкращі значення обох метрик отримано за таких гіперпараметрів: кількість прихованих факторів `n_factors=50`, швидкість навчання `lr_all=0.01` та коефіцієнт регуляризації `reg_all=0.05`.

Для моделі `KNNBasic` найкращі значення обох метрик отримано за таких гіперпараметрів: максимальна кількість сусідів `k=40`, мінімальна кількість сусідів `min_k=1`, метрика схожості `cosine` та підхід на основі схожості користувачів (`user_based=True`).

Порівняння отриманих результатів показало, що модель `SVD` забезпечує нижчі значення `RMSE` (0,9452 проти 1,0265) та `MAE` (0,7452 проти 0,8142) порівняно з `KNNBasic`. Це свідчить про вищу точність прогнозування оцінок фільмів моделлю `SVD` на етапі перехресної перевірки.

**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | 0,9452 | 0,7452 | `n_factors=50`, `lr_all=0.01`, `reg_all=0.05` |
| KNNBasic | 1,0265 | 0,8142 | `k=40`, `min_k=1`, `name='cosine'`, `user_based=True` |

_Порівняйте якість моделей на перехресній перевірці. Зазначте, які параметри стали найкращими._

---

Порівняння результатів перехресної перевірки показало, що модель `SVD` забезпечила кращу точність прогнозування оцінок фільмів, ніж `KNNBasic`. Для `SVD` отримано значення `RMSE = 0,9452` та `MAE = 0,7452`, тоді як для `KNNBasic` ці показники становлять 1,0265 та 0,8142 відповідно. Нижчі значення обох метрик свідчать про менші помилки прогнозування моделі `SVD`.

Для моделі `SVD` найкращими виявилися параметри `n_factors=50`, `lr_all=0.01` та `reg_all=0.05`, а для `KNNBasic` – `k=40`, `min_k=1`, метрика схожості `cosine` та підхід на основі схожості користувачів (`user_based=True`).

Отже, за результатами кросвалідації модель `SVD` виявилася ефективнішою, тому її можна розглядати як перспективнішу для прогнозування оцінок фільмів.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [28]:
# TODO 6.1 — Отримайте best_svd_params і best_knn_params за CV RMSE.
# TODO 6.2 — Створіть trainset із train_data.
# TODO 6.3 — Навчіть best_svd і best_knn.
# Для відтворюваності задайте random_state=42 для SVD.

# Ваш код:
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

trainset = train_data.build_full_trainset()

best_svd = SVD(**best_svd_params, random_state = 42)
best_knn = KNNBasic(**best_knn_params)

best_svd.fit(trainset)
best_knn.fit(trainset)

print('Остаточні моделі успішно навчені.')

print('\nМодель SVD:')
print('Найкращі гіперпараметри:', best_svd_params)

print('\nМодель KNNBasic:')
print('Найкращі гіперпараметри:', best_knn_params)

print('\nКількість оцінок у навчальній вибірці:', trainset.n_ratings)

Computing the cosine similarity matrix...
Done computing similarity matrix.
Остаточні моделі успішно навчені.

Модель SVD:
Найкращі гіперпараметри: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

Модель KNNBasic:
Найкращі гіперпараметри: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}

Кількість оцінок у навчальній вибірці: 80000


###**Пояснення:**

За допомогою атрибута `best_params['rmse']` отримуються оптимальні налаштування кожної моделі за метрикою `RMSE`. Метод `build_full_trainset()` формує повний навчальний набір даних у форматі бібліотеки `Surprise`.

На основі отриманих гіперпараметрів створюються нові екземпляри моделей `SVD` та `KNNBasic`. Для моделі `SVD` додатково встановлюється параметр `random_state=42`, який забезпечує відтворюваність результатів навчання. За допомогою методу `fit(trainset)` обидві моделі навчаються на всій навчальній вибірці. Під час навчання `KNNBasic` обчислюється матриця косинусної схожості між користувачами, що підтверджується відповідними повідомленнями у результатах виконання коду.

Для моделі `SVD` використано найкращі гіперпараметри: кількість прихованих факторів `n_factors=50`, швидкість навчання `lr_all=0.01` та коефіцієнт регуляризації `reg_all=0.05`. Для моделі `KNNBasic` обрано максимальну кількість сусідів `k=40`, мінімальну кількість сусідів `min_k=1`, метрику схожості `cosine` та підхід на основі схожості користувачів (`user_based=True`).

Навчання обох моделей здійснено на 80 000 оцінках фільмів, що відповідає повному обсягу навчальної вибірки.

## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [29]:
# TODO 7.1 — Отримайте svd_predictions та knn_predictions.
# TODO 7.2 — Обчисліть svd_rmse, svd_mae, knn_rmse, knn_mae.
# Підказка: accuracy.rmse(predictions, verbose=False)
#           accuracy.mae(predictions, verbose=False)

# Ваш код:
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

svd_rmse = accuracy.rmse(svd_predictions, verbose = False)
svd_mae = accuracy.mae(svd_predictions, verbose = False)

knn_rmse = accuracy.rmse(knn_predictions, verbose = False)
knn_mae = accuracy.mae(knn_predictions, verbose = False)

print('Результати моделі SVD:')
print('RMSE:', round(svd_rmse, 4))
print('MAE:', round(svd_mae, 4))

print('\nРезультати моделі KNNBasic:')
print('RMSE:', round(knn_rmse, 4))
print('MAE:', round(knn_mae, 4))

print('\nКількість прогнозів SVD:', len(svd_predictions))
print('Кількість прогнозів KNNBasic:', len(knn_predictions))

Результати моделі SVD:
RMSE: 0.9192
MAE: 0.7234

Результати моделі KNNBasic:
RMSE: 1.012
MAE: 0.7984

Кількість прогнозів SVD: 20000
Кількість прогнозів KNNBasic: 20000


###**Пояснення:**

За допомогою методу `test()` обидві моделі формують прогнози оцінок фільмів для тестових записів, які не використовувалися під час навчання та підбору гіперпараметрів. Для оцінювання якості прогнозування застосовуються функції `accuracy.rmse()` та `accuracy.mae()` із бібліотеки `Surprise`, які обчислюють корінь із середньоквадратичної помилки та середню абсолютну помилку відповідно. Отримані значення округлюються до чотирьох знаків після коми за допомогою функції `round()`. Додатково визначається кількість сформованих прогнозів для кожної моделі.

Для моделі `SVD` отримано `RMSE = 0,9192` та `MAE = 0,7234`, тоді як для моделі `KNNBasic` – `RMSE = 1,0120` та `MAE = 0,7984`. Обидві моделі сформували по 20 000 прогнозів, що відповідає кількості записів у незалежній тестовій вибірці. Нижчі значення обох метрик у моделі `SVD` свідчать про менші помилки прогнозування оцінок фільмів порівняно з `KNNBasic`.

In [30]:
# TODO 7.3 — Сформуйте DataFrame results із колонками:
# Model, RMSE, MAE.
# Відсортуйте за RMSE та покажіть порівняння.

# Ваш код:
results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae]
})

results = results.sort_values(by = 'RMSE').reset_index(drop = True)

print('Порівняння точності рекомендаційних моделей:')
display(results.round(4))

print('\nНайкраща модель за RMSE:', results.iloc[0]['Model'])

Порівняння точності рекомендаційних моделей:


,Model,RMSE,MAE
0,SVD,0.9192,0.7234
1,KNNBasic,1.0120,0.7984



Найкраща модель за RMSE: SVD


###**Пояснення:**

За допомогою `pd.DataFrame()` формується таблиця `results`, яка містить назви моделей (`Model`) та відповідні значення метрик `RMSE` і `MAE`. Метод `sort_values(by='RMSE')` використовується для сортування моделей за зростанням середньоквадратичної помилки, а `reset_index(drop=True)` оновлює індекси рядків після сортування. Значення метрик округлюються до чотирьох знаків після коми, а за допомогою `iloc[0]['Model']` визначається модель із найменшим значенням `RMSE`.

За результатами порівняння встановлено, що найкращою моделлю за метрикою `RMSE` є `SVD`, для якої отримано `RMSE = 0,9192` та `MAE = 0,7234`. Модель `KNNBasic` продемонструвала вищі значення помилок: `RMSE = 1,0120` та `MAE = 0,7984`. Таким чином, `SVD` забезпечила кращу точність прогнозування за обома метриками, що підтверджує результати попередньої перехресної перевірки.

Отже, модель `SVD` виявилася точнішою за `KNNBasic` як під час кросвалідації, так і на незалежній тестовій вибірці. Отримані результати підтверджують її ефективність для прогнозування оцінок фільмів у наборі даних `MovieLens 100K`.

**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | 0,9192 | 0,7234 |
| KNNBasic | 1,0120 | 0,7984 |

**Порівняння з CV:** _Прокоментуйте, чи суттєво відрізняються оцінки на кросвалідації та незалежній тестовій вибірці._

---

Порівняння результатів кросвалідації та оцінювання на незалежній тестовій вибірці показало, що отримані значення метрик `RMSE` та `MAE` відрізняються незначно.

Для моделі `SVD` під час кросвалідації отримано `RMSE = 0,9452` та `MAE = 0,7452`, тоді як на тестовій вибірці ці показники становлять 0,9192 та 0,7234 відповідно. Для моделі `KNNBasic` значення `RMSE` зменшилося з 1,0265 до 1,0120, а `MAE` – з 0,8142 до 0,7984.

Таким чином, обидві моделі продемонстрували дещо нижчі помилки прогнозування на незалежній тестовій вибірці порівняно з результатами кросвалідації. Незначні відмінності між показниками свідчать про узгодженість результатів оцінювання та відсутність явних ознак суттєвого перенавчання моделей.

Отже, модель `SVD` зберегла перевагу над `KNNBasic` за обома метриками, що підтверджує результати кросвалідації та свідчить про її кращу здатність прогнозувати оцінки фільмів на досліджуваному наборі даних.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [31]:
# Початковий ID можна змінити
user_id = '196'

# TODO 8.1 — Визначте найкращу за CV RMSE модель.
# TODO 8.2 — Перевірте, що користувач присутній у trainset.
# TODO 8.3 — Знайдіть неоцінені ним фільми.
# TODO 8.4 — Спрогнозуйте оцінки та покажіть топ-10.
# Бажані колонки результату: item_id, predicted_rating.
# Передбачте ситуацію, коли обраного user_id немає у trainset.

# Ваш код:
if grid_search_svd.best_score['rmse'] < grid_search_knn.best_score['rmse']:
    best_model = best_svd
    best_model_name = 'SVD'
else:
    best_model = best_knn
    best_model_name = 'KNNBasic'

print('Найкраща модель за CV RMSE:', best_model_name)

try:
    inner_user_id = trainset.to_inner_uid(user_id)
    print('Користувача', user_id, 'знайдено у навчальній вибірці.')

    rated_items = {
        item_id for item_id, rating in trainset.ur[inner_user_id]
    }

    all_items = set(trainset.all_items())
    unrated_items = all_items - rated_items

    print('Кількість оцінених фільмів:', len(rated_items))
    print('Кількість неоцінених фільмів:', len(unrated_items))

    recommendations = []

    for inner_item_id in unrated_items:
        raw_item_id = trainset.to_raw_iid(inner_item_id)
        predicted_rating = best_model.predict(
            uid = user_id,
            iid = raw_item_id
        ).est

        recommendations.append((raw_item_id, predicted_rating))

    recommendations_df = pd.DataFrame(
        recommendations,
        columns = ['item_id', 'predicted_rating']
    )

    recommendations_df = (
        recommendations_df
        .sort_values(by = 'predicted_rating', ascending = False)
        .head(10)
        .reset_index(drop = True)
    )

    print('\nТоп-10 рекомендацій для користувача', user_id, ':')
    display(recommendations_df.round(4))

except ValueError:
    print('Користувача', user_id, 'не знайдено у навчальній вибірці.')
    print('Виберіть інший user_id.')

Найкраща модель за CV RMSE: SVD
Користувача 196 знайдено у навчальній вибірці.
Кількість оцінених фільмів: 29
Кількість неоцінених фільмів: 1624

Топ-10 рекомендацій для користувача 196 :


,item_id,predicted_rating
0,320,4.4904
1,64,4.4707
2,480,4.4137
3,313,4.4019
4,513,4.3996
5,483,4.3981
6,318,4.3921
7,656,4.3854
8,22,4.3774
9,272,4.3760


###**Пояснення:**

За допомогою методу `trainset.to_inner_uid()` перевіряється наявність вибраного користувача в навчальному наборі та виконується перетворення його зовнішнього ідентифікатора на внутрішній. Після цього за допомогою `trainset.ur` визначається множина фільмів, які користувач уже оцінив. Метод `trainset.all_items()` використовується для отримання всіх фільмів навчального набору, а різниця множин дозволяє визначити фільми, які ще не були оцінені користувачем.

Для кожного неоціненого фільму виконується перетворення внутрішнього ідентифікатора на зовнішній за допомогою методу `trainset.to_raw_iid()`. Після цього метод `predict()` обчислює прогнозовану оцінку фільму, а атрибут `.est` дозволяє отримати її числове значення. Отримані прогнози зберігаються в таблиці `pandas.DataFrame` зі стовпцями `item_id` та `predicted_rating`, сортуються за спаданням прогнозованого рейтингу за допомогою `sort_values()`, після чого методом `head(10)` відбираються 10 фільмів із найвищими прогнозованими оцінками. Додатково передбачено обробку ситуації, коли вибраний користувач відсутній у навчальному наборі.

Користувач із ідентифікатором 196 присутній у навчальній вибірці. Він уже оцінив 29 фільмів, тоді як 1624 фільми залишаються неоціненими в межах навчального набору, який містить 1653 унікальні фільми.

Найвищий прогнозований рейтинг отримав фільм з ідентифікатором 320 – 4,4904, а найнижчий серед десяти рекомендованих – фільм з ідентифікатором 272 із прогнозованою оцінкою 4,3760. Усі рекомендовані фільми мають прогнозовані рейтинги вище 4,3 за п'ятибальною шкалою, що свідчить про високий рівень передбачуваної зацікавленості користувача в цих фільмах.

## 9. Перевірка виконання

Позначте після реалізації:

- [X] Дані `ml-100k` завантажено та перевірено.
- [X] Є розбиття 80/20 з фіксованим `random_state`.
- [X] `Reader`, `train_data` і `testset` створені коректно.
- [X] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [X] Для SVD і KNNBasic показано параметри та результати CV.
- [X] Моделі навчено на повній навчальній частині.
- [X] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [X] Наведено порівняльну таблицю та інтерпретацію.
- [X] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [X] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** _Напишіть 5–8 речень з опорою на отримані числові результати._

---
**Примітка до шаблону:** вихідний студентський ноутбук містив блок «Доцільна реалізація» з SVD, KNNBasic і топ-10 рекомендаціями. Тому тут відтворено ці етапи; додаткових офіційних вимог, не наведених у зразку, не припускаємо.

---

**1) Яку модель обрано за кросвалідацією і чому?**

За результатами трикратної кросвалідації (`cv=3`) обрано рекомендаційну модель `SVD`, оскільки вона продемонструвала менші значення помилок прогнозування порівняно з `KNNBasic`. Для `SVD` отримано `CV RMSE = 0,9452` та `CV MAE = 0,7452`, тоді як для `KNNBasic` ці показники становили 1,0265 та 0,8142 відповідно. Отже, модель `SVD` забезпечила вищу точність прогнозування оцінок фільмів на навчальних даних.

**2) Які найкращі гіперпараметри отримано для обох моделей?**

Для моделі `SVD` найкращими виявилися такі гіперпараметри: кількість прихованих факторів `n_factors=50`, швидкість навчання `lr_all=0.01` та коефіцієнт регуляризації `reg_all=0.05`. Для моделі `KNNBasic` оптимальними стали максимальна кількість сусідів `k=40`, мінімальна кількість сусідів `min_k=1`, метрика схожості `cosine` та підхід на основі схожості користувачів (`user_based=True`). Для обох моделей найкращі комбінації гіперпараметрів за `RMSE` та `MAE` збіглися.

**3) Які підсумкові значення `RMSE` та `MAE` на тестовій вибірці?**

За результатами оцінювання на незалежній тестовій вибірці, яка містила 20 000 оцінок фільмів, модель `SVD` отримала `RMSE = 0,9192` та `MAE = 0,7234`. Для моделі `KNNBasic` відповідні значення становили `RMSE = 1,0120` та `MAE = 0,7984`. Таким чином, `SVD` продемонструвала кращу точність прогнозування за обома метриками, що підтвердило результати попередньої кросвалідації.

**4) Чи є різниця між результатами `CV` та тестування?**

Порівняння результатів кросвалідації та незалежного тестування показало незначні відмінності між значеннями метрик. Для моделі `SVD` показник `RMSE` зменшився з 0,9452 до 0,9192, а `MAE` – з 0,7452 до 0,7234. Для моделі `KNNBasic` значення `RMSE` зменшилося з 1,0265 до 1,0120, а `MAE` – з 0,8142 до 0,7984. Отже, обидві моделі продемонстрували дещо нижчі помилки на тестовій вибірці, а невеликі розбіжності між результатами свідчать про узгодженість оцінювання та відсутність явних ознак суттєвого перенавчання.

**5) Які обмеження мають отримані рекомендації?**

Основним обмеженням побудованої рекомендаційної системи є проблема холодного старту, яка виникає для нових користувачів і фільмів, відсутніх у навчальному наборі. Для таких випадків модель не має достатньої інформації про попередні оцінки та вподобання, що може знижувати точність персоналізованих рекомендацій. Крім того, рекомендації формуються лише для фільмів, представлених у навчальному наборі, та базуються на історії оцінювання без урахування жанрів, описів або інших характеристик фільмів. Для користувача з ідентифікатором 196 було сформовано топ-10 рекомендацій серед 1624 неоцінених ним фільмів, причому прогнозовані рейтинги рекомендованих фільмів перебували в діапазоні від 4,3760 до 4,4904.


###**Підсумкові висновки:**

За результатами трикратної кросвалідації обрано модель `SVD`, оскільки вона показала нижчі значення `RMSE` (0,9452) та `MAE` (0,7452) порівняно з `KNNBasic` (1,0265 та 0,8142 відповідно). `

Найкращими гіперпараметрами для `SVD` стали `n_factors=50`, `lr_all=0.01`, `reg_all=0.05`, а для `KNNBasic` – `k=40`, `min_k=1`, метрика схожості `cosine` та `user_based=True`.

На незалежній тестовій вибірці модель `SVD` також продемонструвала кращу точність прогнозування з `RMSE = 0,9192` та `MAE = 0,7234`, тоді як для `KNNBasic` отримано `RMSE = 1,0120` та `MAE = 0,7984`.

Значення помилок на тестовій вибірці виявилися дещо нижчими, ніж під час кросвалідації, що свідчить про узгодженість результатів і відсутність явних ознак суттєвого перенавчання.

На основі моделі `SVD` сформовано топ-10 рекомендацій для користувача 196 серед 1624 неоцінених ним фільмів, причому прогнозовані рейтинги перебувають у діапазоні від 4,3760 до 4,4904.

Основними обмеженнями рекомендаційної системи є проблема холодного старту для нових користувачів і фільмів, а також залежність точності прогнозування від наявності попередніх оцінок.

Таким чином, `SVD` забезпечила кращу точність прогнозування, ніж `KNNBasic`, і була обрана для формування персоналізованих рекомендацій фільмів.

##**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**

**Реалізація власного рекомендаційного алгоритму**

In [32]:
# Вибираємо користувача з ідентифікатором 196.
# Знаходимо інших користувачів, які оцінювали ті самі фільми.
# Обчислюємо схожість користувачів за їхніми оцінками за допомогою кореляції Пірсона.
# Визначаємо 30 найбільш схожих користувачів.
# Прогнозуємо рейтинги фільмів, які вибраний користувач ще не оцінював.
# Формуємо топ-10 фільмів із найбільшими прогнозованими оцінками.

# Підготовка даних
ratings = train_df.copy()
ratings['user_id'] = ratings['user_id'].astype(str)
ratings['item_id'] = ratings['item_id'].astype(str)
ratings['rating'] = ratings['rating'].astype(float)

# Створення словника оцінок користувачів
user_ratings = {}

for user, group in ratings.groupby('user_id'):
    user_ratings[user] = dict(zip(group['item_id'], group['rating']))

# Середні оцінки користувачів і фільмів
user_means = ratings.groupby('user_id')['rating'].mean().to_dict()
item_means = ratings.groupby('item_id')['rating'].mean().to_dict()

# Загальна середня оцінка
global_mean = ratings['rating'].mean()

# Обчислення схожості двох користувачів
def user_similarity(user1, user2):
    common_items = set(user_ratings[user1]) & set(user_ratings[user2])

    if len(common_items) < 3:
        return 0.0

    ratings1 = np.array([user_ratings[user1][i] for i in common_items])
    ratings2 = np.array([user_ratings[user2][i] for i in common_items])

    ratings1 = ratings1 - ratings1.mean()
    ratings2 = ratings2 - ratings2.mean()

    denominator = np.linalg.norm(ratings1) * np.linalg.norm(ratings2)

    if denominator == 0:
        return 0.0

    similarity = np.dot(ratings1, ratings2) / denominator

    # Зменшення впливу невеликої кількості спільних оцінок
    similarity *= len(common_items) / (len(common_items) + 10)

    return similarity

# Формування рекомендацій
def recommend_movies(user_id, top_n = 10, k = 30):

    user_id = str(user_id)

    if user_id not in user_ratings:
        print('Користувача не знайдено у навчальній вибірці.')
        return pd.DataFrame(columns = ['item_id', 'predicted_rating'])

    # Обчислення схожості з іншими користувачами
    similarities = []

    for other_user in user_ratings:
        if other_user != user_id:
            sim = user_similarity(user_id, other_user)

            if sim > 0:
                similarities.append((other_user, sim))

    # Вибір найбільш схожих користувачів
    similarities.sort(key = lambda x: x[1], reverse = True)
    neighbors = similarities[:k]

    # Фільми, які користувач уже оцінив
    rated_items = set(user_ratings[user_id])

    # Усі фільми навчальної вибірки
    all_items = set(item_means)

    # Неоцінені фільми
    unrated_items = all_items - rated_items

    recommendations = []

    for item in unrated_items:
        numerator = 0.0
        denominator = 0.0

        for neighbor, similarity in neighbors:
            if item in user_ratings[neighbor]:
                deviation = user_ratings[neighbor][item] - user_means[neighbor]

                numerator += similarity * deviation
                denominator += abs(similarity)

        if denominator > 0:
            predicted_rating = user_means[user_id] + numerator / denominator
        else:
            predicted_rating = (
                user_means[user_id] + item_means.get(item, global_mean)
            ) / 2

        # Обмеження оцінки діапазоном від 1 до 5
        predicted_rating = np.clip(predicted_rating, 1, 5)

        recommendations.append((item, predicted_rating))

    # Створення таблиці рекомендацій
    recommendations_df = pd.DataFrame(
        recommendations,
        columns = ['item_id', 'predicted_rating']
    )

    recommendations_df = (
        recommendations_df
        .sort_values(
            by = ['predicted_rating', 'item_id'],
            ascending = [False, True]
        )
        .head(top_n)
        .reset_index(drop = True)
    )

    print('Користувач:', user_id)
    print('Кількість оцінених фільмів:', len(rated_items))
    print('Кількість неоцінених фільмів:', len(unrated_items))
    print('Кількість схожих користувачів:', len(neighbors))

    return recommendations_df

# Формування топ-10 рекомендацій
custom_recommendations = recommend_movies('196', top_n = 10, k = 30)

print('\nТоп-10 рекомендацій власного алгоритму:')
display(custom_recommendations.round(4))

Користувач: 196
Кількість оцінених фільмів: 29
Кількість неоцінених фільмів: 1624
Кількість схожих користувачів: 30

Топ-10 рекомендацій власного алгоритму:


,item_id,predicted_rating
0,1639,5.0000
1,361,5.0000
2,624,5.0000
3,481,4.9716
4,344,4.9664
5,1449,4.9482
6,1512,4.9482
7,1558,4.9482
8,1139,4.8883
9,1311,4.8883


###**Пояснення:**

За допомогою функції `user_similarity()` визначається схожість між користувачами на основі кореляції Пірсона. Для обчислення враховуються фільми, які оцінили обидва користувачі, причому мінімальна кількість спільних оцінок становить три. Додатково застосовується коефіцієнт зменшення схожості для пар користувачів із невеликою кількістю спільно оцінених фільмів.

Функція `recommend_movies()` знаходить найбільш схожих користувачів, визначає фільми, які вибраний користувач ще не оцінював, та прогнозує їхні рейтинги на основі оцінок сусідів з урахуванням відхилень від середніх оцінок. Для фільмів, щодо яких недостатньо інформації від схожих користувачів, використовується резервний прогноз на основі середніх оцінок користувача та фільму. Прогнозовані значення обмежуються діапазоном від 1 до 5, після чого сортуються за спаданням та відбираються 10 фільмів із найвищими рейтингами.

Користувач 196 оцінив 29 фільмів, а 1624 фільми залишаються неоціненими в навчальній вибірці. Для формування рекомендацій алгоритм визначив 30 найбільш схожих користувачів.

Найвищі прогнозовані оцінки отримали фільми з ідентифікаторами 1639, 361 та 624, для яких прогнозований рейтинг становить 5,0000. Решта рекомендованих фільмів мають рейтинги від 4,8883 до 4,9716. Усі 10 рекомендацій сформовано серед фільмів, які вибраний користувач ще не оцінював у навчальній вибірці.

**Оцінювання точності власного рекомендаційного алгоритму та порівняння моделей**

In [33]:
# Збереження найближчих сусідів для кожного користувача
neighbors_cache = {}

def get_neighbors(user_id, k = 30):
    if user_id not in neighbors_cache:
        similarities = []

        for other_user in user_ratings:
            if other_user != user_id:
                sim = user_similarity(user_id, other_user)

                if sim > 0:
                    similarities.append((other_user, sim))

        similarities.sort(key = lambda x: x[1], reverse = True)
        neighbors_cache[user_id] = similarities[:k]

    return neighbors_cache[user_id]

# Прогнозування оцінки одного фільму
def predict_custom(user_id, item_id):
    user_id = str(user_id)
    item_id = str(item_id)

    # Якщо користувач відсутній у навчальній вибірці
    if user_id not in user_ratings:
        return item_means.get(item_id, global_mean)

    neighbors = get_neighbors(user_id)

    numerator = 0.0
    denominator = 0.0

    for neighbor, similarity in neighbors:
        if item_id in user_ratings[neighbor]:
            deviation = user_ratings[neighbor][item_id] - user_means[neighbor]

            numerator += similarity * deviation
            denominator += abs(similarity)

    if denominator > 0:
        predicted_rating = user_means[user_id] + numerator / denominator
    else:
        predicted_rating = (
            user_means[user_id] + item_means.get(item_id, global_mean)
        ) / 2

    return float(np.clip(predicted_rating, 1, 5))

custom_predictions = []

for row in test_df.itertuples(index = False):
    prediction = predict_custom(row.user_id, row.item_id)
    custom_predictions.append(prediction)

# Фактичні оцінки
actual_ratings = test_df['rating'].to_numpy(dtype = float)

# Обчислення RMSE та MAE
custom_predictions = np.array(custom_predictions)

custom_rmse = np.sqrt(np.mean((actual_ratings - custom_predictions) ** 2))
custom_mae = np.mean(np.abs(actual_ratings - custom_predictions))

print('Результати власного рекомендаційного алгоритму:')
print('RMSE:', round(custom_rmse, 4))
print('MAE:', round(custom_mae, 4))
print('Кількість прогнозів:', len(custom_predictions))

# Порівняння трьох моделей
comparison = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic', 'Власний алгоритм'],
    'RMSE': [svd_rmse, knn_rmse, custom_rmse],
    'MAE': [svd_mae, knn_mae, custom_mae]
})

comparison = comparison.sort_values(
    by = 'RMSE'
).reset_index(drop = True)

print('\nПорівняння точності трьох рекомендаційних алгоритмів:')
display(comparison.round(4))

Результати власного рекомендаційного алгоритму:
RMSE: 1.0111
MAE: 0.7826
Кількість прогнозів: 20000

Порівняння точності трьох рекомендаційних алгоритмів:


,Model,RMSE,MAE
0,SVD,0.9192,0.7234
1,Власний алгоритм,1.0111,0.7826
2,KNNBasic,1.0120,0.7984


###**Пояснення:**

Для оптимізації обчислень створюється словник `neighbors_cache`, у якому зберігаються результати пошуку найбільш схожих користувачів. Функція `get_neighbors()` визначає 30 найближчих сусідів на основі попередньо реалізованої функції `user_similarity()`, яка обчислює схожість користувачів за допомогою кореляції Пірсона.

За допомогою функції `predict_custom()` здійснюється прогнозування оцінок фільмів на основі рейтингів схожих користувачів з урахуванням відхилень від їхніх середніх оцінок. Якщо інформації для прогнозування недостатньо, використовується резервний прогноз на основі середніх оцінок користувача та фільму. Для користувачів, відсутніх у навчальній вибірці, передбачено використання середньої оцінки фільму або загальної середньої оцінки. Прогнозовані рейтинги обмежуються діапазоном від 1 до 5.

Для кожного запису тестової вибірки за допомогою функції `predict_custom()` обчислюється прогнозована оцінка, після чого за допомогою бібліотеки `NumPy` визначаються метрики `RMSE` та `MAE`. Отримані результати порівнюються з показниками раніше навчених моделей `SVD` та `KNNBasic`. Для наочного представлення результатів створюється таблиця `pandas.DataFrame`, яка містить назви моделей і значення метрик, відсортовані за зростанням `RMSE`.

Власний рекомендаційний алгоритм сформував 20 000 прогнозів для незалежної тестової вибірки. Отримано значення `RMSE = 1,0111` та `MAE = 0,7826`.

За результатами порівняння встановлено, що найкращу точність прогнозування забезпечила модель `SVD`, яка має найменші значення `RMSE` та `MAE`. Власний рекомендаційний алгоритм продемонстрував дещо кращі результати, ніж `KNNBasic`: значення `RMSE` виявилося меншим на 0,0009, а `MAE` — на 0,0158. Водночас власний алгоритм поступився моделі `SVD` за обома метриками.

Отже, власний рекомендаційний алгоритм забезпечив точність прогнозування, близьку до `KNNBasic`, та дещо перевершив її за `RMSE` і `MAE` на досліджуваній тестовій вибірці.

###**Загальний висновок до додаткового завдання:**

Власний рекомендаційний алгоритм на основі колаборативної фільтрації користувачів із використанням кореляції Пірсона дозволив визначити найбільш схожих користувачів та сформувати персоналізовані рекомендації фільмів без використання готових рекомендаційних моделей. Для користувача з ідентифікатором 196 на основі оцінок 30 найближчих сусідів сформовано топ-10 рекомендацій серед 1624 неоцінених фільмів із прогнозованими рейтингами від 4,8883 до 5,0000. Оцінювання власного алгоритму на незалежній тестовій вибірці показало `RMSE = 1,0111` та `MAE = 0,7826`, що є дещо кращими результатами порівняно з `KNNBasic (RMSE = 1,0120; MAE = 0,7984)`, проте поступається точності моделі `SVD (RMSE = 0,9192; MAE = 0,7234)`. Отримані результати свідчать, що власний алгоритм забезпечує якість прогнозування, близьку до `KNNBasic`, однак `SVD` залишається найточнішою серед трьох досліджуваних моделей.